# 🟡 Medium: Classifier-Free Guidance

Implement **classifier-free guidance** (CFG), the single knob behind "prompt adherence" in every
modern text-to-image and text-to-video model.

### Core Idea

Guided diffusion originally needed a separate classifier $p(c \mid x)$ to push samples toward a
condition. CFG removes it: train **one** network with the condition randomly dropped (say 10% of the
time replaced by a null token $\varnothing$), so it learns the conditional *and* the unconditional
score at once. Then at sampling time, use Bayes:

$$\nabla_x \log p(x \mid c) + w'\,\nabla_x \log \frac{p(x \mid c)}{p(x)}$$

which in $\epsilon$-space is just a linear extrapolation away from the unconditional prediction:

$$\hat{\epsilon} = \epsilon_\varnothing + w \cdot (\epsilon_c - \epsilon_\varnothing)$$

| $w$ | meaning |
|:---:|---|
| 0 | ignore the condition entirely |
| 1 | plain conditional sampling — **no guidance** |
| 5–15 | typical; sharper, more prompt-faithful, less diverse |

Note it costs **2× compute**: every step evaluates the model twice (usually as one batch of size 2B).

**The catch — and the rescale fix.** Extrapolating with $w \gg 1$ inflates the variance of
$\hat{\epsilon}$, which over-saturates and blows out images. *Common Diffusion Noise Schedules and
Sample Steps are Flawed* fixes it by renormalising back to the conditional branch's statistics and
blending:

$$\hat{\epsilon}_\text{rescaled} = \hat{\epsilon}\cdot\frac{\sigma(\epsilon_c)}{\sigma(\hat{\epsilon})}, \qquad \hat{\epsilon}_\text{final} = \phi\,\hat{\epsilon}_\text{rescaled} + (1-\phi)\,\hat{\epsilon}$$

Standard deviations are **per sample** (over all non-batch dims) — batch elements must never mix.

### Signature
```python
def classifier_free_guidance(eps_uncond, eps_cond, guidance_scale, rescale=0.0):
    # eps_uncond: model output for the null condition, shape (B, ...)
    # eps_cond:   model output for the real condition, same shape
    # guidance_scale: w  (1.0 == no guidance)
    # rescale: phi in [0, 1]; 0.0 disables the std-rescale trick
    # returns: guided prediction, same shape
    ...
```

### Rules
- Use `torch.std` with its default (unbiased) correction, computed per sample with `keepdim=True`
- `guidance_scale=1.0` must return `eps_cond` **exactly**; `0.0` must return `eps_uncond`
- `rescale=0.0` must be a no-op

### Example
```
eps_u = torch.randn(4, 3, 8, 8)
eps_c = torch.randn(4, 3, 8, 8)
out = classifier_free_guidance(eps_u, eps_c, 7.5)
# -> eps_u + 7.5 * (eps_c - eps_u)
out = classifier_free_guidance(eps_u, eps_c, 7.5, rescale=1.0)
# -> same direction, but each sample's std matches eps_c's std
```

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def classifier_free_guidance(eps_uncond, eps_cond, guidance_scale, rescale=0.0):
    # eps_uncond / eps_cond: (B, ...)   guidance_scale: w   rescale: phi in [0, 1]
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
eps_u = torch.randn(4, 3, 8, 8)
eps_c = torch.randn(4, 3, 8, 8)

out = classifier_free_guidance(eps_u, eps_c, 7.5)
print("w=7.5 matches formula :", torch.allclose(out, eps_u + 7.5 * (eps_c - eps_u), atol=1e-6))
print("w=1.0 is conditional  :", torch.allclose(classifier_free_guidance(eps_u, eps_c, 1.0), eps_c, atol=1e-6))
print("std blow-up (w=7.5)   :", out.std(dim=(1, 2, 3)).mean().item(), "vs cond", eps_c.std(dim=(1, 2, 3)).mean().item())

out_r = classifier_free_guidance(eps_u, eps_c, 7.5, rescale=1.0)
print("after rescale=1.0     :", out_r.std(dim=(1, 2, 3)).mean().item())

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("cfg")